# Eigenvalues and eigenvectors basics

1. Limitations of root finding for the characteristic polynomial (Wilkinson's polynomial)
2. Library routines: `eigh` and `eig`
3. Power iteration
4. Householder reflections
5. QR iteration

In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

rng = np.random.default_rng(1)

## 1. Wilkinson's polynomial

The eigenvalues of $\mathbf{A}$ are the roots of $p(\lambda)=\det(\mathbf{A}-\lambda\mathbf{I})$. So why not compute the coefficients of $p$ and use a root finder?

Take a simple-looking eigenvalue problem: a diagonal matrix, $\mathbf{A}=\text{diag}(1, 2, \dots, 20)$, with

$$
p(\lambda)=(\lambda-1)(\lambda-2)\cdots(\lambda-20)=\lambda^{20}-210\lambda^{19}+\dots+20!
$$

We find the roots twice: once from the coefficients rounded to double precision, and once after Wilkinson's perturbation $-210 \to -210-2^{-23}$ (a relative change of $\sim 6\times10^{-10}$ in one coefficient).

In [ ]:
eigs = np.arange(1, 21)
coeffs = np.poly(eigs)               # coefficients of (x-1)(x-2)...(x-20), in floating point
roots_rounded = np.roots(coeffs)     # only error: rounding the coefficients to doubles
coeffs_wilk = coeffs.copy()
coeffs_wilk[1] -= 2.0**-23           # Wilkinson's perturbation of the x^19 coefficient
roots_wilk = np.roots(coeffs_wilk)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(eigs, np.zeros_like(eigs), "kx", ms=10, mew=2, label="true eigenvalues")
ax.plot(roots_rounded.real, roots_rounded.imag, "o", mfc="none", ms=9,
        label="roots of rounded coefficients")
ax.plot(roots_wilk.real, roots_wilk.imag, "s", ms=6,
        label=r"roots after $-210 \rightarrow -210 - 2^{-23}$")
ax.set(xlabel=r"Re $\lambda$", ylabel=r"Im $\lambda$")
ax.legend(fontsize=10, loc="upper left")
plt.show()

print("largest error, rounded coefficients:", np.max(np.abs(np.sort(roots_rounded.real) - eigs)))
print("number of complex roots after perturbation:", np.sum(np.abs(roots_wilk.imag) > 1e-6))

Going through $p(\lambda)$ turns a perfectly conditioned problem into a badly conditioned one. Compare with an eigenvalue routine applied directly to the matrix:

In [ ]:
print(np.linalg.eigvalsh(np.diag(eigs.astype(float))))

Fun fact: `np.roots` works the other way around. It builds the *companion matrix*, whose characteristic polynomial is $p$, and computes its eigenvalues.

## 2. Library routines

| matrix | `numpy` | `scipy.linalg` | LAPACK |
|---|---|---|---|
| symmetric / Hermitian | `np.linalg.eigh` | `eigh` (also generalized $\mathbf{K}\mathbf{x}=\lambda\mathbf{M}\mathbf{x}$) | `_syevd`, `_heevd` |
| general | `np.linalg.eig` | `eig` | `_geev` |

If your matrix is symmetric, use `eigh`: it is faster, the eigenvalues come out real and sorted, and the eigenvectors are orthonormal.

Throughout, we use the $3\times3$ matrix from the 1D finite-difference Laplacian:

In [ ]:
A = np.array([[ 2., -1.,  0.],
              [-1.,  2., -1.],
              [ 0., -1.,  2.]])

w, X = np.linalg.eigh(A)
print("eigenvalues:", w)              # exact: 2 - sqrt(2), 2, 2 + sqrt(2)
print("eigenvectors (columns):\n", X)
print("max |A X - X diag(w)| =", np.max(np.abs(A @ X - X * w)))
print("max |X^T X - 1|       =", np.max(np.abs(X.T @ X - np.eye(3))))

## 3. Power iteration

Start from a random $\mathbf{v}_0$ and iterate $\mathbf{v}_{k+1}=\mathbf{A}\mathbf{v}_k/|\mathbf{A}\mathbf{v}_k|$. The vector converges to the eigenvector $\mathbf{x}_0$ with the **largest** $|\lambda|$, and the **Rayleigh quotient** $\lambda \approx \mathbf{v}_k^T\mathbf{A}\mathbf{v}_k$ (with $|\mathbf{v}_k|=1$) gives the eigenvalue.

The error shrinks by $|\lambda_1/\lambda_0|$ per step for the eigenvector, and $|\lambda_1/\lambda_0|^2$ for the eigenvalue (for symmetric $\mathbf{A}$).

In [ ]:
lam0, x0 = w[-1], X[:, -1]          # exact answer, for comparison
ratio = abs(w[-2] / w[-1])

v = rng.normal(size=3)
v /= np.linalg.norm(v)
ks, vec_err, val_err = [], [], []
for k in range(1, 31):
    v = A @ v
    v /= np.linalg.norm(v)
    lam = v @ A @ v                  # Rayleigh quotient
    ks.append(k)
    vec_err.append(np.linalg.norm(v - np.sign(v @ x0) * x0))   # eigenvectors are defined up to sign
    val_err.append(abs(lam - lam0))
ks = np.array(ks)
print(f"after {ks[-1]} iterations: lambda = {lam:.15f}, exact = {lam0:.15f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(ks, vec_err, "o", label="eigenvector error")
ax.semilogy(ks, val_err, "s", label="eigenvalue error")
ax.semilogy(ks, vec_err[0] * ratio ** (ks - 1), "k--", label=r"$\propto |\lambda_1/\lambda_0|^k$")
ax.semilogy(ks, val_err[0] * ratio ** (2 * (ks - 1)), "k:", label=r"$\propto |\lambda_1/\lambda_0|^{2k}$")
ax.set(xlabel="iteration $k$", ylim=(1e-17, 10))
ax.legend(fontsize=10)
plt.show()

**Try this:**

- **Inverse iteration:** iterate with $\mathbf{A}^{-1}$ instead (use `np.linalg.solve(A, v)`, not `inv`!). Which eigenvector does it find? How fast?
- **Shifted inverse iteration:** iterate with $(\mathbf{A}-\sigma\mathbf{I})^{-1}$ for $\sigma=1.9$. Which eigenvector, and how fast now?

## 4. Householder reflections

The Householder matrix $\mathbf{P}=\mathbf{I}-2\mathbf{u}\mathbf{u}^T/|\mathbf{u}|^2$ reflects through the plane perpendicular to $\mathbf{u}$. Choosing $\mathbf{u}=\mathbf{x}-|\mathbf{x}|\mathbf{e}_0$ maps $\mathbf{x}$ onto $|\mathbf{x}|\mathbf{e}_0$, zeroing all but its first component.

In [ ]:
def householder(x):
    # Return P such that P @ x = |x| e_0.
    u = x.astype(float).copy()
    u[0] -= np.linalg.norm(x)
    return np.eye(len(x)) - 2 * np.outer(u, u) / (u @ u)

x = rng.normal(size=5)
P = householder(x)
print("x     =", x)
print("P x   =", P @ x)
print("|x|   =", np.linalg.norm(x))
print("max |P^T P - 1| =", np.max(np.abs(P.T @ P - np.eye(5))))
print("max |P - P^T|   =", np.max(np.abs(P - P.T)))

(In practice one uses $\mathbf{u}=\mathbf{x}+\text{sign}(x_0)|\mathbf{x}|\mathbf{e}_0$, mapping to $-\text{sign}(x_0)|\mathbf{x}|\mathbf{e}_0$, to avoid cancellation when $\mathbf{x}$ is nearly parallel to $\mathbf{e}_0$.)

Applying $N-2$ such reflections as similarity transformations, $\mathbf{A}\to\mathbf{P}\mathbf{A}\mathbf{P}$, reduces a symmetric matrix to **tridiagonal** form with the same eigenvalues. `scipy.linalg.hessenberg` does exactly this (for a general matrix it gives upper Hessenberg form):

In [ ]:
B = rng.normal(size=(6, 6))
B = B + B.T                                   # random symmetric matrix
T, Q = scipy.linalg.hessenberg(B, calc_q=True)
np.set_printoptions(precision=3, suppress=True)
print(T)
np.set_printoptions()
print("eigenvalues of B:", np.linalg.eigvalsh(B))
print("eigenvalues of T:", np.linalg.eigvalsh(T))

## 5. QR iteration

Factor $\mathbf{A}_k=\mathbf{Q}_k\mathbf{R}_k$, then set $\mathbf{A}_{k+1}=\mathbf{R}_k\mathbf{Q}_k=\mathbf{Q}_k^T\mathbf{A}_k\mathbf{Q}_k$. Each step is a similarity transformation, so the eigenvalues don't change, and for a symmetric matrix $\mathbf{A}_k$ converges to a diagonal matrix.

We build a symmetric $5\times5$ matrix with known eigenvalues $5, 4, 2.5, 1.5, 0.5$, and watch the off-diagonal elements decay.

In [ ]:
Qr, _ = np.linalg.qr(rng.normal(size=(5, 5)))
evals = np.array([5.0, 4.0, 2.5, 1.5, 0.5])
A5 = Qr @ np.diag(evals) @ Qr.T           # symmetric, known eigenvalues

snapshots = {}
Ak = A5.copy()
for k in range(61):
    if k in (0, 5, 20, 60):
        snapshots[k] = Ak.copy()
    Q, R = np.linalg.qr(Ak)
    Ak = R @ Q

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (k, M) in zip(axes, snapshots.items()):
    im = ax.imshow(np.abs(M) + 1e-16, norm=LogNorm(1e-12, 10), cmap="viridis")
    ax.set_title(f"$|\\mathbf{{A}}_{{{k}}}|$", fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()
print("diagonal of A_60:", np.round(np.diag(Ak), 6), "  exact:", evals)

The element $(\mathbf{A}_k)_{ij}$ below the diagonal decays like $|\lambda_i/\lambda_j|^k$, so the pair $(4, 5)$ converges slowest. Real implementations first reduce to tridiagonal/Hessenberg form (section 4), which makes each QR step cheap, and use **shifts** to make convergence much faster.

**Try this:** change the eigenvalues to $5, 4.9, 2.5, 1.5, 0.5$. How many iterations do you need now?